<div style="display:flex; align-items:flex-start; gap:2rem; padding:1rem 0;">

  <div style="flex-shrink:0;">
    <img src="https://cdn.oreillystatic.com/images/sitewide-headers/oreilly_logo_mark_red.svg"
         style="height:36px; display:block; margin-bottom:0.75rem;"/>
    <img src="https://i.imgur.com/ITL8dZE.jpeg"
         style="width:260px; border-radius:4px; box-shadow:0 2px 8px rgba(0,0,0,0.15); display:block;"/>
  </div>

  <div style="flex:1; font-family:sans-serif;">
    <h1 style="font-size:1.6rem; font-weight:600; margin:0 0 0.25rem 0;">
      AI, ML and GenAI in the Lakehouse
    </h1>
   Name:          chapter 04-03-Model-Building

   Author:    Bennie Haelen
   Date:      10-09-2024

   Purpose:   This notebook performs model training and evaluation for the Hotel Bookings
              cancellation prediction use case.

      An outline of the different sections in this notebook:
        1 - Read the feature-engineered Delta table from Unity Catalog
        2 - Prepare training and test data
              2-1 Define the feature matrix (X) and target variable (y)
              2-2 Split the data into training and test sets
              2-3 Apply standard scaling to the feature matrix
              2-4 Visualize feature distributions after scaling
        3 - Model training with MLflow tracking
              3-1 Set the MLflow experiment
              3-2 Logistic Regression
              3-3 K-Neighbors Classifier
              3-4 Decision Tree Classifier
              3-5 Random Forest Classifier
        4 - Compare model results and register the best model to Unity Catalog

        <div>

In [0]:
%pip install --upgrade threadpoolctl==3.5.0

In [0]:
%restart_python

##### Perform the required imports
The primary libraries used in this notebook are:
- NumPy for numeric arrays
- Pandas for DataFrames
- Matplotlib and Seaborn for visualization
- Scikit-learn for model training, scaling, and evaluation
- MLflow for experiment tracking and model registration

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

## Set basic options for Pandas and Matplotlib

In [0]:
pd.set_option("display.max_columns", None)
%matplotlib inline
plt.style.use('fivethirtyeight')

## Read the feature-engineered Delta table from Unity Catalog

We read the feature table produced by the feature engineering notebook directly from
Unity Catalog. This gives us a clean, governed, versioned input for model training
without relying on raw file paths.

In [0]:
# Unity Catalog coordinates - update these to match your setup
catalog = "book_ai_ml_lakehouse"
schema  = "default"
table   = "hotel_bookings_features"

# Read the managed Delta table as a Spark DataFrame, then convert to pandas
spark_df = spark.read.table(f"{catalog}.{schema}.{table}")
bookings_df = spark_df.toPandas()

print(f"Loaded {bookings_df.shape[0]:,} rows and {bookings_df.shape[1]} columns from {catalog}.{schema}.{table}")
bookings_df.head()

## Prepare training and test data

### Define the feature matrix (X) and target variable (y)

The target variable `is_canceled` was encoded as 0 (not canceled) and 1 (canceled)
during feature engineering. We separate it from the feature matrix here.

In [0]:
# Feature matrix: all columns except the target
X = bookings_df.drop(columns=['is_canceled'])

# Target variable
y = bookings_df['is_canceled']

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable distribution:\n{y.value_counts()}")

### Split the data into training and test sets (80/20)

In [0]:
# Split into 80% training and 20% test data.
# random_state=42 ensures reproducibility across runs.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set:  X={X_train.shape}, y={y_train.shape}")
print(f"Test set:      X={X_test.shape},  y={y_test.shape}")

### Apply standard scaling to the feature matrix

StandardScaler transforms each feature to have a mean of zero and a standard deviation
of one. This is important for distance-based models such as K-Neighbors and for
Logistic Regression, which is sensitive to feature scale.

A critical rule: fit the scaler only on the training data, then apply the same
transformation to the test data. Fitting on the full dataset would constitute data
leakage, allowing information from the test set to influence the scaling parameters.

In [0]:
scaler = StandardScaler()

# Fit on training data only, then transform
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to the test data using training statistics
X_test_scaled = scaler.transform(X_test)

### Visualize feature distributions after scaling

In [0]:
# Identify the most informative features to visualize using variance and
# correlation with the target variable
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)

feature_variance    = X_train_scaled_df.var()
feature_correlation = X_train_scaled_df.corrwith(y_train.reset_index(drop=True))

top_variance_features    = feature_variance.nlargest(5).index
top_correlation_features = feature_correlation.abs().nlargest(5).index
selected_features        = list(set(top_variance_features).union(set(top_correlation_features)))

num_features = len(selected_features)
rows = (num_features // 2) + 1
cols = 2

plt.figure(figsize=(16, 4 * rows))

for i, feature in enumerate(selected_features):
    plt.subplot(rows, cols, i + 1)
    sns.kdeplot(X_train_scaled_df[feature], color='orange', fill=True, label='Scaled')
    plt.axvline(X_train_scaled_df[feature].mean(), color='red', linestyle='--', label='Mean')
    plt.title(f'Density plot of {feature} (scaled)', fontsize=14, pad=20)
    plt.xlabel(feature, fontsize=12)
    plt.legend()

plt.tight_layout(pad=3.0)
plt.show()

## Model training with MLflow tracking

### Set the MLflow experiment

We use `mlflow.set_experiment()` to direct all runs to a named experiment in the
Databricks workspace. Using `dbutils` to retrieve the current username keeps the
path dynamic and avoids hardcoding personal credentials into the notebook.

In [0]:
# Retrieve the current Databricks username dynamically
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_path = f"/Users/{username}/hotel_bookings_cancellations_prediction"

mlflow.set_experiment(experiment_name=experiment_path)
print(f"MLflow experiment: {experiment_path}")

### MLflow autologging

We enable `mlflow.sklearn.autolog()` once before all training runs. This automatically
captures model parameters, training metrics, feature importances, and the serialized
model artifact for every scikit-learn model trained within an active run, without
requiring manual `mlflow.log_metric()` calls for each model.

Additional artifacts such as confusion matrices and classification reports are logged
manually where autologging does not capture them.

In [0]:
# Enable autologging for all scikit-learn runs in this session
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True)

### Logistic Regression

Logistic Regression is a strong baseline for binary classification. It is fast to train,
highly interpretable, and provides well-calibrated probability estimates. The
`class_weight='balanced'` parameter adjusts for any class imbalance in the cancellation
labels, and `max_iter=10000` ensures convergence on this feature-rich dataset.

In [0]:
with mlflow.start_run(run_name='logistic_regression') as run_lr:

    model_lr = LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)
    model_lr.fit(X_train_scaled, y_train)

    y_pred_lr = model_lr.predict(X_test_scaled)

    acc_lr = accuracy_score(y_test, y_pred_lr)
    clf_report_lr = classification_report(y_test, y_pred_lr)

    # Log artifacts not captured by autolog
    mlflow.log_text(str(confusion_matrix(y_test, y_pred_lr)), "confusion_matrix.txt")
    mlflow.log_text(clf_report_lr, "classification_report.txt")

    run_id_lr = run_lr.info.run_id
    print(f"Run ID: {run_id_lr}")
    print(f"Accuracy: {acc_lr:.4f}")
    print(f"Classification Report:\n{clf_report_lr}")

### K-Neighbors Classifier

K-Nearest Neighbors classifies each observation based on the majority class among its
k nearest neighbors in the feature space. It requires no explicit training phase but is
sensitive to feature scale, which is why we apply StandardScaler before this step.
With `n_neighbors=5`, each prediction is based on the five closest training examples.

In [0]:
with mlflow.start_run(run_name='k_neighbors_classifier') as run_knn:

    model_knn = KNeighborsClassifier(n_neighbors=5, weights='uniform')
    model_knn.fit(X_train_scaled, y_train)

    y_pred_knn = model_knn.predict(X_test_scaled)

    acc_knn = accuracy_score(y_test, y_pred_knn)
    clf_report_knn = classification_report(y_test, y_pred_knn)

    mlflow.log_text(str(confusion_matrix(y_test, y_pred_knn)), "confusion_matrix.txt")
    mlflow.log_text(clf_report_knn, "classification_report.txt")

    run_id_knn = run_knn.info.run_id
    print(f"Run ID: {run_id_knn}")
    print(f"Accuracy: {acc_knn:.4f}")
    print(f"Classification Report:\n{clf_report_knn}")

### Decision Tree Classifier

A Decision Tree partitions the feature space using a series of binary splits, making it
highly interpretable. The `max_depth=5` constraint prevents the tree from memorizing the
training data, acting as a form of regularization. This model is useful as a baseline
for understanding which features drive splits early in the tree.

In [0]:
with mlflow.start_run(run_name='decision_tree_classifier') as run_dt:

    model_dt = DecisionTreeClassifier(max_depth=5, random_state=42)
    model_dt.fit(X_train_scaled, y_train)

    y_pred_dt = model_dt.predict(X_test_scaled)

    acc_dt = accuracy_score(y_test, y_pred_dt)
    clf_report_dt = classification_report(y_test, y_pred_dt)

    mlflow.log_text(str(confusion_matrix(y_test, y_pred_dt)), "confusion_matrix.txt")
    mlflow.log_text(clf_report_dt, "classification_report.txt")

    run_id_dt = run_dt.info.run_id
    print(f"Run ID: {run_id_dt}")
    print(f"Accuracy: {acc_dt:.4f}")
    print(f"Classification Report:\n{clf_report_dt}")

### Random Forest Classifier

Random Forest builds an ensemble of decision trees, each trained on a random subset of
the data and features, then aggregates their predictions by majority vote. This reduces
the variance that a single deep decision tree would exhibit. With `n_estimators=100`,
we train 100 trees; `max_depth=10` limits tree depth to balance performance and
training time.

In [0]:
with mlflow.start_run(run_name='random_forest_classifier') as run_rf:

    model_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model_rf.fit(X_train_scaled, y_train)

    y_pred_rf = model_rf.predict(X_test_scaled)

    acc_rf = accuracy_score(y_test, y_pred_rf)
    clf_report_rf = classification_report(y_test, y_pred_rf)

    mlflow.log_text(str(confusion_matrix(y_test, y_pred_rf)), "confusion_matrix.txt")
    mlflow.log_text(clf_report_rf, "classification_report.txt")

    run_id_rf = run_rf.info.run_id
    print(f"Run ID: {run_id_rf}")
    print(f"Accuracy: {acc_rf:.4f}")
    print(f"Classification Report:\n{clf_report_rf}")

## Compare model results

After training all four models, we compare their accuracy scores side by side to
identify the best candidate for registration and deployment.

In [0]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'K-Neighbors Classifier',
        'Decision Tree Classifier',
        'Random Forest Classifier'
    ],
    'Accuracy': [acc_lr, acc_knn, acc_dt, acc_rf],
    'Run ID': [run_id_lr, run_id_knn, run_id_dt, run_id_rf]
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print(results.to_string(index=False))

# Identify the best model
best_row = results.iloc[0]
print(f"\nBest model: {best_row['Model']} (Accuracy: {best_row['Accuracy']:.4f})")

## Register the best model to Unity Catalog

Registering the best-performing model in Unity Catalog brings the same governance
benefits we apply to data assets: versioning, lineage, access control, and a clear
promotion path from development to production. The three-level namespace
`catalog.schema.model` keeps models co-located with the feature data they were
trained on.

In [0]:
# Unity Catalog model name - three-level namespace
model_uc_path = f"{catalog}.{schema}.hotel_cancellation_classifier"

# Register the best run's model artifact to Unity Catalog
best_run_id = best_row['Run ID']
model_uri   = f"runs:/{best_run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_uc_path
)

print(f"Model registered: {model_uc_path}")
print(f"Version: {registered_model.version}")
print(f"Source run: {best_run_id}")

## End of notebook